# 10 · Psychometrics & summary stats

The **model-free atlas**: what categorisation looks like in every phase we ran,
per animal, shown side by side — straight from choices and stimuli, no fitted
decision model. Establishes the phenomena that 3X/4X fit and 2X perturbs.

**Display, not contrast.** All trials, no filtering. Cross-phase differences are
*shown*, not attributed — the opto-vs-masking / on-vs-off decomposition is 2X's
job, and 'which internal model' is 4X's. No genotype contrast here.

**Cohort-split, not pooled.** The first cohort (SS01–13) has laser-free `regular`
Hard-A/Hard-B — the only clean hard-distribution behaviour, and the HMM/SLDS
cohort. The opto cohort (SS14–23) has only opto/masking there. Different
data-generating conditions, so the fold is organised *within* cohort; nothing is
merged across them.

In [1]:
from shared_setup import *
apply_style()

experiment, info = load_data()

first_ids = [a for a in FIRST_COHORT if a in experiment.animals]
opto_ids  = [a for a in OPTO_COHORT  if a in experiment.animals]
print(f"first cohort present ({len(first_ids)}): {first_ids}")
print(f"opto cohort present  ({len(opto_ids)}): {opto_ids}")
missing = [a for a in FIRST_COHORT + OPTO_COHORT if a not in experiment.animals]
if missing: print(f"hardcoded but absent (check IDs): {missing}")

NameError: name 'apply_style' is not defined

In [ ]:
# phases per cohort (default presets). tail=N keeps the last N sessions.
FIRST_PHASES = [
    ('uniform expert', 'expert_uniform', 5),
    ('hard-a',         'hard_a_regular', None),
    ('hard-b',         'hard_b_regular', None),
]
OPTO_PHASES = [
    ('uniform expert', 'expert_uniform',  5),
    ('uniform masking','uniform_masking', None),
    ('uniform opto',   'uniform_opto',    None),
    ('hard-a opto',    'hard_a_opto',     None),
    ('hard-a masking', 'hard_a_masking',  None),
    ('hard-b opto',    'hard_b_opto',     None),
    ('hard-b masking', 'hard_b_masking',  None),
]
COHORTS = [('first', first_ids, FIRST_PHASES), ('opto', opto_ids, OPTO_PHASES)]

# 'psychometric' expands to mu, sigma, lapse_low, lapse_high
STATS = ['accuracy', 'hard_accuracy', 'easy_accuracy', 'side_bias', 'recency', 'psychometric']

def phase_sessions(animal, preset, tail):
    ph = select_sessions(animal, preset)
    if tail and ph:
        ph = sorted(ph, key=lambda s: s.session_idx)[-tail:]
    return ph

## §2 · One animal, end to end

**SS16** (opto cohort) — its full behavioural profile: the psychometric per phase
(per-session faint + overall bold, all trials), the summary stats per phase, and
the update matrix per phase. The atlas in §3 is these same calls over every animal.

In [ ]:
ex = experiment.get_animal('SS16')
ex_phases = [(l, p, t) for (l, p, t) in OPTO_PHASES if phase_sessions(ex, p, t)]

# psychometric per phase
ncols = min(4, len(ex_phases)); nrows = int(np.ceil(len(ex_phases) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(3.3 * ncols, 3.3 * nrows), squeeze=False, sharey=True)
axf = axes.ravel()
for ax, (label, preset, tail) in zip(axf, ex_phases):
    ph = phase_sessions(ex, preset, tail)
    plot_psychometric(compute_psychometric(ph, mode='per_session', n_bootstrap=0), ax=ax,
                      show_individual=True, show_data=False)
    plot_psychometric(compute_psychometric(ph, mode='pooled', n_bootstrap=0), ax=ax,
                      color=get_colour(3), linewidth=2.5, show_ci=False, show_data=False)
    ax.set_title(f"{label} ({len(ph)} sess)", fontsize=9)
for ax in axf[len(ex_phases):]: fig.delaxes(ax)
fig.suptitle('SS16 — psychometric per phase', fontsize=12); fig.tight_layout()

In [ ]:
# summary stats per phase (pooled, all trials)
rows = []
for label, preset, tail in ex_phases:
    pooled = compute_stat(phase_sessions(ex, preset, tail), STATS, mode='pooled')['pooled']
    rows.append({'phase': label, **{k: round(v, 3) for k, v in pooled.items()}})
pd.DataFrame(rows).set_index('phase')

In [ ]:
# update matrix per phase
ncols = min(4, len(ex_phases)); nrows = int(np.ceil(len(ex_phases) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 3.2 * nrows), squeeze=False)
axf = axes.ravel()
for ax, (label, preset, tail) in zip(axf, ex_phases):
    plot_um(compute_um(phase_sessions(ex, preset, tail)), ax=ax); ax.set_title(label, fontsize=9)
for ax in axf[len(ex_phases):]: fig.delaxes(ax)
fig.suptitle('SS16 — update matrix per phase', fontsize=12); fig.tight_layout()

## §3 · Atlas fold

Every animal, every phase it has: the pooled scalars → tidy
`{animal, cohort, phase, stat, value}`. All-trials, no filtering — these are
behaviour as collected, not off-filtered baselines (2X/4X filter where they must).
Order-dependent readouts (the UM) stay out of the fold; they are shown, not tabulated.

In [ ]:
atlas_rows = []
for coh, ids, phases in COHORTS:
    for aid in ids:
        animal = experiment.get_animal(aid)
        for label, preset, tail in phases:
            ph = phase_sessions(animal, preset, tail)
            if not ph:
                continue
            pooled = compute_stat(ph, STATS, mode='pooled')['pooled']
            atlas_rows += collect_rows([{'stat': k, 'value': float(v)} for k, v in pooled.items()],
                                       animal=aid, group=coh, group_col='cohort', phase=label)
atlas_df = pd.DataFrame(atlas_rows)
print(f"{atlas_df['animal'].nunique()} animals, {atlas_df['phase'].nunique()} phases, "
      f"{atlas_df['stat'].nunique()} stats, {len(atlas_df)} rows")
atlas_df.head()

## §4 · Cohort views

How much animals vary in each phase — descriptive, within cohort. No phase-vs-phase
test. Three views: the per-animal psychometrics overlaid, the scalar spread across
animals, and the cohort-mean update matrix.

In [ ]:
# per-animal pooled psychometrics overlaid, per phase (the spread band)
for coh, ids, phases in COHORTS:
    present = [(l, p, t) for (l, p, t) in phases if any(phase_sessions(experiment.get_animal(a), p, t) for a in ids)]
    if not present: continue
    fig, axes = plt.subplots(1, len(present), figsize=(3.2 * len(present), 3.3), squeeze=False, sharey=True)
    for ax, (label, preset, tail) in zip(axes[0], present):
        n = 0
        for aid in ids:
            ph = phase_sessions(experiment.get_animal(aid), preset, tail)
            if ph:
                plot_psychometric(compute_psychometric(ph, mode='pooled', n_bootstrap=0), ax=ax,
                                  color='0.4', alpha=0.4, linewidth=1.2, show_data=False, show_ci=False)
                n += 1
        ax.set_title(f"{label} (n={n})", fontsize=9)
    fig.suptitle(f"{coh} cohort — per-animal psychometrics", fontsize=12); fig.tight_layout()

In [ ]:
# scalar spread across animals, per phase (points = animals, connected within animal)
DISPLAY_STATS = ['accuracy', 'mu', 'sigma', 'side_bias', 'recency']

def phase_groups(df, labels):
    # {phase: frame with columns stat, value, session(=animal)} for plot_session_stats_single
    return {l: df[df['phase'] == l].rename(columns={'animal': 'session'})
            for l in labels if (df['phase'] == l).any()}

for coh, ids, phases in COHORTS:
    cdf = atlas_df[atlas_df['cohort'] == coh]
    if cdf.empty: continue
    labels = [l for (l, _, _) in phases]
    groups = phase_groups(cdf, labels)
    fig, axes = plt.subplots(1, len(DISPLAY_STATS), figsize=(3.2 * len(DISPLAY_STATS), 3.4), squeeze=False)
    for ax, stat in zip(axes[0], DISPLAY_STATS):
        plot_session_stats_single(groups, stat, ax=ax, connect=True, show_session_ids=True)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right', fontsize=7)
    fig.suptitle(f"{coh} cohort — scalar spread across animals", fontsize=12); fig.tight_layout()

In [ ]:
# cohort-mean update matrix per phase (min_sources guards sparse cells)
for coh, ids, phases in COHORTS:
    present = [(l, p, t) for (l, p, t) in phases if any(phase_sessions(experiment.get_animal(a), p, t) for a in ids)]
    if not present: continue
    fig, axes = plt.subplots(1, len(present), figsize=(3.2 * len(present), 3.2), squeeze=False)
    for ax, (label, preset, tail) in zip(axes[0], present):
        ums = [compute_um(phase_sessions(experiment.get_animal(a), preset, tail))
               for a in ids if phase_sessions(experiment.get_animal(a), preset, tail)]
        if ums:
            plot_um(average_um(ums, min_sources=2), ax=ax)
        ax.set_title(f"{label} (n={len(ums)})", fontsize=9)
    fig.suptitle(f"{coh} cohort — mean update matrix per phase", fontsize=12); fig.tight_layout()

## §5 · Summary + emit

The per-animal all-trials scalars, for reference. Read-only otherwise.

In [ ]:
# try:
#     out = FIG_DIR.parent / 'results'; out.mkdir(parents=True, exist_ok=True)
#     atlas_df.to_csv(out / '10_atlas_scalars.csv', index=False)
#     print(f'wrote atlas ({len(atlas_df)} rows) to {out}')
# except Exception as e:
#     print(f'emit skipped: {e}')